# Vector Database Comparison
## ChromaDB, Pinecone, and Weaviate for Stripe Documentation RAG

### Objectives
1. **Setup and configure** three different vector databases
- ChromaDB setup (local, easy development)
- Pinecone setup (serverless, production-ready)
- Weaviate setup (GraphQL, hybrid search)
2. **Generate embeddings** efficiently using OpenAI's API
3. **Ingest data** and benchmark performance
4. **Compare query performance** across databases
- Performance benchmarking framework
- Retrieval quality comparison
- Cost analysis
5. **Implement hybrid search** (dense + sparse retrieval)
6. **Test metadata filtering** capabilities
7. **Evaluate retrieval quality** with test queries


### Prerequisites
- Chunked data available in `./stripe_docs_data/chunks/`
- OpenAI API key for embeddings
- (Optional) Pinecone API key for cloud deployment

### Environment Setup

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
import json
from openai import OpenAI
import hashlib
import time
from tqdm.auto import tqdm
import tiktoken
import chromadb
from chromadb.config import Settings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from pinecone import Pinecone, ServerlessSpec
    PINECONE_AVAILABLE = True
except ImportError:
    PINECONE_AVAILABLE = False
    print("⚠️ Pinecone not available - will skip Pinecone tests")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("✓ Environment setup complete")
print(f"  OpenAI API: {'✓' if os.getenv('OPENAI_API_KEY') else '✗'}")
print(f"  Pinecone API: {'✓' if os.getenv('PINECONE_API_KEY') else '⚠️ Optional'}")
print(f"  ChromaDB: ✓ (local)")

### Configuration & Data Models

In [ ]:
# Configuration
CONFIG = {
    'data_dir': Path('./stripe_docs_data'),
    'chunks_dir': Path('./stripe_docs_data/chunks'),
    'output_dir': Path('./stripe_docs_data/vector_dbs'),

    'chunking_strategy': 'recursive',  # Options: fixed_size, recursive, semantic, sentence_window, hierarchical
    
    # Embedding configuration
    'embedding_model': 'text-embedding-3-small', #Option 'text-embedding-ada-002'
    'embedding_dimension': 1536,
    'batch_size': 100,  # For embedding generation
    
    # Database configuration
    'chromadb_path': './stripe_docs_data/vector_dbs/chromadb',
    'collection_name': 'stripe_docs_rag',
    
    # Pinecone configuration
    'pinecone_index_name': 'stripe-docs-rag',
    'pinecone_cloud': 'aws',
    'pinecone_region': 'us-east-1',
    
    # Query configuration
    'top_k': 5,  # Number of results to retrieve
    
    # Testing
    'test_sample_size': 50,  # Number of chunks to use for testing
}

# Create output directory
CONFIG['output_dir'].mkdir(exist_ok=True, parents=True)

print("✓ Configuration loaded")
print(f"  Data directory: {CONFIG['data_dir']}")
print(f"  Chunking strategy: {CONFIG['chunking_strategy']}")
print(f"  Embedding model: {CONFIG['embedding_model']}")
print(f"  Embedding dimension: {CONFIG['embedding_dimension']}")

In [ ]:
@dataclass
class BenchmarkResult:
    """Store benchmark results for comparison"""
    database: str
    operation: str  # 'ingestion', 'query', 'hybrid_search', etc.
    duration: float  # seconds
    count: int  # number of items processed
    throughput: float  # items per second
    memory_mb: Optional[float] = None
    metadata: Dict = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}

@dataclass
class QueryResult:
    """Store query results for evaluation"""
    query: str
    database: str
    results: List[Dict]
    duration: float
    scores: List[float]
    metadata: Dict = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}

print("✓ Data models defined")

### Load Chunked Data

In [ ]:
def load_chunks_for_strategy(strategy: str, db_format: str = 'chromadb') -> Dict:
    """
    Load chunks.
    
    Args:
        strategy: Chunking strategy name (e.g., 'recursive', 'semantic')
        db_format: Database format ('chromadb', 'pinecone', 'weaviate')
    
    Returns:
        Dict with documents, metadatas, and ids
    """
    file_path = CONFIG['chunks_dir'] / f"{db_format}_{strategy}.json"
    
    if not file_path.exists():
        raise FileNotFoundError(
            f"Chunks file not found: {file_path}\n"
        )
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"✓ Loaded {len(data.get('documents', data)) if isinstance(data, dict) else len(data)} chunks from {file_path.name}")
    return data

# Load chunks using configured strategy
print(f"Loading chunks for strategy: {CONFIG['chunking_strategy']}")
chunks_data = load_chunks_for_strategy(CONFIG['chunking_strategy'], 'chromadb')

# Extract components
documents = chunks_data['documents']
metadatas = chunks_data['metadatas']
ids = chunks_data['ids']

print(f"\nData Summary:")
print(f"  Total chunks: {len(documents)}")
print(f"  Sample ID: {ids[0]}")
print(f"  Sample metadata keys: {list(metadatas[0].keys())}")
print(f"  Sample document length: {len(documents[0])} chars")

# Show sample
print(f"\nSample chunk content:")
print(f"  ID: {ids[0]}")
print(f"  Type: {metadatas[0].get('doc_type', 'N/A')}")
print(f"  Category: {metadatas[0].get('category', 'N/A')}")
print(f"  Content preview: {documents[0][:200]}...")

### Embedding Generation

We generate each embedding only once. We use the embedding from chache if available. We save the generated embeddings in cache.

In [ ]:
class EmbeddingGenerator:
    """Generate embeddings with batching and caching"""
    
    def __init__(self, model: str = 'text-embedding-3-small', batch_size: int = 100):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = model
        self.batch_size = batch_size
        self.cache_file = CONFIG['output_dir'] / f'embeddings_cache_{model}.json'
        self.cache = self._load_cache()
        
    def _load_cache(self) -> Dict:
        """Load cached embeddings if available"""
        if self.cache_file.exists():
            with open(self.cache_file, 'r') as f:
                cache = json.load(f)
            print(f"✓ Loaded {len(cache)} cached embeddings")
            return cache
        return {}
    
    def _save_cache(self):
        """Save embeddings cache"""
        with open(self.cache_file, 'w') as f:
            json.dump(self.cache, f)
    
    def _get_cache_key(self, text: str) -> str:
        """Generate cache key for text"""
        return hashlib.md5(text.encode()).hexdigest()
    
    def generate_batch(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for a batch of texts.
        """
        # Check cache
        embeddings = []
        texts_to_embed = []
        cache_indices = []

        # Uses cache when available
        for i, text in enumerate(texts):
            cache_key = self._get_cache_key(text)
            if cache_key in self.cache:
                embeddings.append(self.cache[cache_key])
                cache_indices.append(i)
            else:
                texts_to_embed.append(text)
        
        # Generate new embeddings
        if texts_to_embed:
            response = self.client.embeddings.create(
                input=texts_to_embed,
                model=self.model
            )
            
            new_embeddings = [item.embedding for item in response.data]
            
            # Cache new embeddings
            for text, embedding in zip(texts_to_embed, new_embeddings):
                cache_key = self._get_cache_key(text)
                self.cache[cache_key] = embedding
            
            # Merge with cached
            result = []
            new_idx = 0
            for i in range(len(texts)):
                if i in cache_indices:
                    result.append(embeddings[cache_indices.index(i)])
                else:
                    result.append(new_embeddings[new_idx])
                    new_idx += 1
            
            embeddings = result
        
        return embeddings
    
    def generate_all(self, texts: List[str], show_progress: bool = True) -> List[List[float]]:
        """
        Generate embeddings for all texts with batching.
        """
        all_embeddings = []
        
        iterator = range(0, len(texts), self.batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Generating embeddings")
        
        for i in iterator:
            batch = texts[i:i + self.batch_size]
            embeddings = self.generate_batch(batch)
            all_embeddings.extend(embeddings)
            
            # Save cache periodically
            if i % (self.batch_size * 10) == 0:
                self._save_cache()
        
        # Final cache save
        self._save_cache()
        
        return all_embeddings

print("✓ Embedding generator defined")

In [ ]:
# Generate embeddings for all chunks
print("Generating embeddings...")
print(f"  Model: {CONFIG['embedding_model']}")
print(f"  Documents: {len(documents)}")
print(f"  Batch size: {CONFIG['batch_size']}")

embedding_generator = EmbeddingGenerator(
    model=CONFIG['embedding_model'],
    batch_size=CONFIG['batch_size']
)

start_time = time.time()
embeddings = embedding_generator.generate_all(documents)
duration = time.time() - start_time

print(f"\n✓ Generated {len(embeddings)} embeddings in {duration:.2f}s")
print(f"  Throughput: {len(embeddings) / duration:.1f} embeddings/second")
print(f"  Embedding shape: {len(embeddings[0])} dimensions")

# Calculate cost estimate
tokenizer = tiktoken.get_encoding('cl100k_base')
total_tokens = sum(len(tokenizer.encode(doc)) for doc in documents)
cost_per_million = 0.02  # $0.02 per 1M tokens for text-embedding-3-small
estimated_cost = (total_tokens / 1_000_000) * cost_per_million

print(f"\nCost Analysis:")
print(f"  Total tokens: {total_tokens:,}")
print(f"  Estimated cost: ${estimated_cost:.4f}")

## ChromaDB Setup and Ingestion

ChromaDB:
- Local development and testing
- Easy setup with Python-native API
- Good performance for small to medium datasets
- No external dependencies

In [ ]:
class ChromaDBManager:
    """Manage ChromaDB operations"""
    
    def __init__(self, persist_directory: str, collection_name: str):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        
        # Initialize client
        self.client = chromadb.PersistentClient(
            path=persist_directory,
            settings=Settings(
                anonymized_telemetry=False, # anonymous usage statistics
                allow_reset=True
            )
        )
        
        # Get or create collection
        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Loaded existing collection: {collection_name}")
        except:
            self.collection = self.client.create_collection(
                name=collection_name,
                metadata={"hnsw:space": "cosine"} # options: l2 (euclidian), ip (inner product)
            )
            print(f"✓ Created new collection: {collection_name}")
    
    def ingest(self, documents: List[str], embeddings: List[List[float]], 
               metadatas: List[Dict], ids: List[str]) -> BenchmarkResult:
        """
        Ingest documents into ChromaDB.
        """
        start_time = time.time()
        clean_metadatas = []
        for meta in metadatas:
            clean_meta = {}
            for key, value in meta.items():
                if isinstance(value, (str, int, float, bool)):
                    clean_meta[key] = value
                elif isinstance(value, list):
                    clean_meta[key] = str(value)  # Convert lists to string
                elif value is None:
                    clean_meta[key] = "none"
                else:
                    clean_meta[key] = str(value)
            clean_metadatas.append(clean_meta)
        
        # Add to collection
        self.collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=clean_metadatas,
            ids=ids
        )
        
        duration = time.time() - start_time
        
        return BenchmarkResult(
            database='ChromaDB',
            operation='ingestion',
            duration=duration,
            count=len(documents),
            throughput=len(documents) / duration,
            metadata={'collection_name': self.collection_name}
        )
    
    def query(self, query_embedding: List[float], n_results: int = 5,
             where: Optional[Dict] = None) -> QueryResult:
        """
        Query ChromaDB for similar documents.
        """
        start_time = time.time()
        
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where
        )
        
        duration = time.time() - start_time
        
        # Format results
        formatted_results = []
        for i in range(len(results['ids'][0])):
            formatted_results.append({
                'id': results['ids'][0][i],
                'document': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })
        
        return QueryResult(
            query="",  # Will be set by caller
            database='ChromaDB',
            results=formatted_results,
            duration=duration,
            scores=[1 - d for d in results['distances'][0]]  # Convert distance to similarity
        )
    
    def get_stats(self) -> Dict:
        """Get collection statistics"""
        return {
            'count': self.collection.count(),
            'name': self.collection.name,
            'metadata': self.collection.metadata
        }

print("✓ ChromaDB manager defined")

In [ ]:
# Initialize ChromaDB
print("Setting up ChromaDB...")
chroma_manager = ChromaDBManager(
    persist_directory=CONFIG['chromadb_path'],
    collection_name=CONFIG['collection_name']
)

# Ingest data
print(f"\nIngesting {len(documents)} documents into ChromaDB...")
chroma_result = chroma_manager.ingest(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"\n✓ ChromaDB ingestion complete")
print(f"  Duration: {chroma_result.duration:.2f}s")
print(f"  Throughput: {chroma_result.throughput:.1f} docs/second")
print(f"  Total documents: {chroma_manager.get_stats()['count']}")

## Pinecone Setup and Ingestion

Pinecone:
- Production deployments
- Serverless scaling
- High availability
- Managed infrastructure

Requires Pinecone API key.

In [ ]:
class PineconeManager:
    """Manage Pinecone operations"""
    
    def __init__(self, index_name: str, dimension: int, cloud: str = 'aws', region: str = 'us-east-1'):
        self.index_name = index_name
        self.dimension = dimension
        
        # Initialize Pinecone
        api_key = os.getenv('PINECONE_API_KEY')
        if not api_key:
            raise ValueError("PINECONE_API_KEY not found in environment")
        
        self.pc = Pinecone(api_key=api_key)
        
        # Create or connect to index
        existing_indexes = [idx.name for idx in self.pc.list_indexes()]
        
        if index_name not in existing_indexes:
            print(f"Creating new Pinecone index: {index_name}")
            self.pc.create_index(
                name=index_name,
                dimension=dimension,
                metric='cosine',
                spec=ServerlessSpec(
                    cloud=cloud,
                    region=region
                )
            )
            print("  Waiting for index to be ready...")
            time.sleep(10)  # Wait for index initialization
        
        self.index = self.pc.Index(index_name)
        print(f"✓ Connected to Pinecone index: {index_name}")
    
    def ingest(self, embeddings: List[List[float]], metadatas: List[Dict], 
               ids: List[str], batch_size: int = 100) -> BenchmarkResult:
        """
        Ingest vectors into Pinecone.
        """
        start_time = time.time()
        
        # Prepare vectors
        vectors = []
        for i, (vec_id, embedding, metadata) in enumerate(zip(ids, embeddings, metadatas)):
            clean_metadata = {}
            for key, value in metadata.items():
                if value is None:
                    # Skip null values
                    continue
                elif isinstance(value, (str, int, float, bool)):
                    clean_metadata[key] = value
                elif isinstance(value, list):
                    # Pinecone accepts lists of strings
                    clean_list = [str(v) for v in value if v is not None]
                    if clean_list:  # Only add if list is not empty
                        clean_metadata[key] = clean_list
                else:
                    # Convert other types to string
                    clean_metadata[key] = str(value)


            # Pinecone stores text in metadata
            vectors.append({
                'id': vec_id,
                'values': embedding,
                'metadata': clean_metadata
            })
        
        # Upsert in batches
        for i in tqdm(range(0, len(vectors), batch_size), desc="Upserting to Pinecone"):
            batch = vectors[i:i + batch_size]
            self.index.upsert(vectors=batch)
        
        duration = time.time() - start_time
        
        return BenchmarkResult(
            database='Pinecone',
            operation='ingestion',
            duration=duration,
            count=len(vectors),
            throughput=len(vectors) / duration,
            metadata={'index_name': self.index_name}
        )
    
    def query(self, query_embedding: List[float], top_k: int = 5,
             filter: Optional[Dict] = None) -> QueryResult:
        """
        Query Pinecone for similar vectors.
        """
        start_time = time.time()
        
        results = self.index.query(
            vector=query_embedding,
            top_k=top_k,
            filter=filter,
            include_metadata=True
        )
        
        duration = time.time() - start_time
        
        # Format results
        formatted_results = []
        scores = []
        for match in results.matches:
            formatted_results.append({
                'id': match.id,
                'metadata': match.metadata,
                'score': match.score
            })
            scores.append(match.score)
        
        return QueryResult(
            query="",
            database='Pinecone',
            results=formatted_results,
            duration=duration,
            scores=scores
        )
    
    def get_stats(self) -> Dict:
        """Get index statistics"""
        stats = self.index.describe_index_stats()
        return {
            'total_vectors': stats.total_vector_count,
            'dimension': stats.dimension,
            'index_fullness': stats.index_fullness
        }

print("✓ Pinecone manager defined")

In [ ]:
# Initialize and use Pinecone (if API key available)
pinecone_result = None
pinecone_manager = None

if PINECONE_AVAILABLE and os.getenv('PINECONE_API_KEY'):
    try:
        print("Setting up Pinecone...")
        pinecone_manager = PineconeManager(
            index_name=CONFIG['pinecone_index_name'],
            dimension=CONFIG['embedding_dimension'],
            cloud=CONFIG['pinecone_cloud'],
            region=CONFIG['pinecone_region']
        )
        
        # Ingest data
        print(f"\nIngesting {len(embeddings)} vectors into Pinecone...")
        pinecone_result = pinecone_manager.ingest(
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids,
            batch_size=100
        )
        
        print(f"\n✓ Pinecone ingestion complete")
        print(f"  Duration: {pinecone_result.duration:.2f}s")
        print(f"  Throughput: {pinecone_result.throughput:.1f} docs/second")
        print(f"  Total vectors: {pinecone_manager.get_stats()['total_vectors']}")
        
    except Exception as e:
        print(f"⚠️ Pinecone setup failed: {e}")
        print("  Continuing without Pinecone...")
else:
    print("⚠️ Pinecone skipped (API key not available or package not installed)")
    print("  To use Pinecone: Add PINECONE_API_KEY to .env file")

## Query Performance Comparison

In [ ]:
# Define test queries
TEST_QUERIES = [
    "How do I create a payment intent?",
    "What is the difference between charges and payment intents?",
    "How do I handle webhooks for subscription events?",
    "What parameters are required for creating a customer?",
    "How do I implement 3D Secure authentication?",
    "What are the best practices for error handling?",
    "How do I set up recurring billing?",
    "What is the refund process?",
]

print(f"Test queries: {len(TEST_QUERIES)}")
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"  {i}. {q}")

In [ ]:
# Generate query embeddings
print("Generating query embeddings...")
query_embeddings = embedding_generator.generate_all(TEST_QUERIES, show_progress=False)
print(f"✓ Generated {len(query_embeddings)} query embeddings")

In [ ]:
# Run queries on all databases
query_results = []

print("\nRunning test queries on all databases...\n")

for i, (query, query_emb) in enumerate(zip(TEST_QUERIES, query_embeddings)):
    print(f"Query {i+1}: {query}")
    
    # ChromaDB
    result = chroma_manager.query(query_emb, n_results=CONFIG['top_k'])
    result.query = query
    query_results.append(result)
    print(f"  ChromaDB: {result.duration*1000:.1f}ms")
    
    # Pinecone
    if pinecone_manager:
        result = pinecone_manager.query(query_emb, top_k=CONFIG['top_k'])
        result.query = query
        query_results.append(result)
        print(f"  Pinecone: {result.duration*1000:.1f}ms")

    print()

print(f"✓ Completed {len(TEST_QUERIES)} queries across databases")

## Performance Analysis and Visualization

In [ ]:
# Compile benchmark results
benchmark_data = []

# Ingestion results
if chroma_result:
    benchmark_data.append(asdict(chroma_result))
if pinecone_result:
    benchmark_data.append(asdict(pinecone_result))

# Query results - aggregate by database
query_stats = {}
for result in query_results:
    if result.database not in query_stats:
        query_stats[result.database] = []
    query_stats[result.database].append(result.duration)

for db_name, durations in query_stats.items():
    benchmark_data.append({
        'database': db_name,
        'operation': 'query',
        'duration': np.mean(durations),
        'count': len(durations),
        'throughput': len(durations) / sum(durations),
        'memory_mb': None,
        'metadata': {
            'avg_duration_ms': np.mean(durations) * 1000,
            'min_duration_ms': np.min(durations) * 1000,
            'max_duration_ms': np.max(durations) * 1000,
            'std_duration_ms': np.std(durations) * 1000
        }
    })

# Create DataFrame
benchmark_df = pd.DataFrame(benchmark_data)

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(benchmark_df[['database', 'operation', 'duration', 'throughput']].to_string(index=False))

# Save results
benchmark_df.to_csv(CONFIG['output_dir'] / 'benchmark_results.csv', index=False)
print(f"\n✓ Saved benchmark results to {CONFIG['output_dir'] / 'benchmark_results.csv'}")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Ingestion throughput
ingestion_data = benchmark_df[benchmark_df['operation'] == 'ingestion']
if not ingestion_data.empty:
    ingestion_data.plot(x='database', y='throughput', kind='bar', ax=axes[0, 0], 
                       color='steelblue', legend=False)
    axes[0, 0].set_title('Ingestion Throughput', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Database')
    axes[0, 0].set_ylabel('Documents/Second')
    axes[0, 0].tick_params(axis='x', rotation=45)

# Ingestion duration
if not ingestion_data.empty:
    ingestion_data.plot(x='database', y='duration', kind='bar', ax=axes[0, 1], 
                       color='coral', legend=False)
    axes[0, 1].set_title('Ingestion Duration', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Database')
    axes[0, 1].set_ylabel('Seconds')
    axes[0, 1].tick_params(axis='x', rotation=45)

# Query latency distribution
query_durations_by_db = {}
for result in query_results:
    if result.database not in query_durations_by_db:
        query_durations_by_db[result.database] = []
    query_durations_by_db[result.database].append(result.duration * 1000)  # Convert to ms

if query_durations_by_db:
    box_data = [query_durations_by_db[db] for db in query_durations_by_db.keys()]
    axes[1, 0].boxplot(box_data, tick_labels=list(query_durations_by_db.keys()))
    axes[1, 0].set_title('Query Latency Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Database')
    axes[1, 0].set_ylabel('Latency (ms)')
    axes[1, 0].tick_params(axis='x', rotation=45)

# Average query latency
query_data = benchmark_df[benchmark_df['operation'] == 'query'].copy()
if not query_data.empty:
    query_data['avg_latency_ms'] = query_data['duration'] * 1000
    query_data.plot(x='database', y='avg_latency_ms', kind='bar', ax=axes[1, 1], 
                   color='lightgreen', legend=False)
    axes[1, 1].set_title('Average Query Latency', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Database')
    axes[1, 1].set_ylabel('Latency (ms)')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Performance visualizations saved")

## Sample Query Results Inspection

In [ ]:
# Display sample results for first query
sample_query = TEST_QUERIES[0]
print(f"Sample Query: {sample_query}")
print("="*80)

# Get results from each database for this query
sample_results = [r for r in query_results if r.query == sample_query]

for result in sample_results:
    print(f"\n{result.database} Results:")
    print("-"*80)
    print(f"Query time: {result.duration*1000:.1f}ms")
    print(f"\nTop 3 results:")
    
    for i, (res, score) in enumerate(zip(result.results[:3], result.scores[:3]), 1):
        print(f"\n  {i}. Score: {score:.4f}")
        if 'metadata' in res:
            meta = res['metadata']
            print(f"     Type: {meta.get('doc_type', 'N/A')}")
            print(f"     Category: {meta.get('category', 'N/A')}")
        
        # Show content preview
        content = res.get('document', res.get('content', ''))
        preview = content[:200] if len(content) > 200 else content
        print(f"     Content: {preview}...")
    
    print()

## 11. Metadata Filtering Performance Test

In [ ]:
# Test metadata filtering on ChromaDB
print("Testing metadata filtering (ChromaDB)...\n")

# Filter by doc_type
test_query_emb = query_embeddings[0]

# Test different filters
filters = [
    {"doc_type": "api_reference"},
    {"category": "payments"},
    {"doc_type": "guide"},
]

filter_results = []

for filter_dict in filters:
    start = time.time()
    result = chroma_manager.query(
        query_embedding=test_query_emb,
        n_results=CONFIG['top_k'],
        where=filter_dict
    )
    duration = time.time() - start
    
    filter_str = str(filter_dict)
    print(f"Filter: {filter_str}")
    print(f"  Results: {len(result.results)}")
    print(f"  Duration: {duration*1000:.1f}ms")
    if result.results:
        print(f"  Sample result type: {result.results[0]['metadata'].get('doc_type', 'N/A')}")
    print()
    
    filter_results.append({
        'filter': filter_str,
        'count': len(result.results),
        'duration_ms': duration * 1000
    })

print("✓ Metadata filtering test complete")